# YOLOX-Nano — PhenoBench multiclass full frames (320×320)

Fine-tunes upstream **YOLOX-Nano** on the PhenoBench full-frame (native 1024x1024) multiclass bundle and
exports a deployment-shaped ONNX graph. This is the YOLOX arm of the
architecture comparison; the SSD arm is notebooks `11`–`14` (fine-tune) and
`21`–`24` (PTQ/QAT).

## Why 320×320

The `ssd-mn2` / `ssd-mn2-fpnlite` arm is trained, quantised and benchmarked at
320×320 throughout. Training YOLOX at the same input resolution keeps the
detector architecture the only free variable; any other choice would confound
architecture with input scale, and the reported deltas would not be
attributable. 320 is also the deployment size on the i.MX targets, so the
network is trained at the resolution it is quantised and benchmarked at.

## Inputs

| role | path |
|---|---|
| image bundle | `/kaggle/input/datasets/freimutdiener/mc-phenobench-yolox/phenobench_multiclass_full_1024_raw` |
| ├ images | `<image bundle>/images` |
| └ provenance | `<image bundle>/raw_images_metadata.json` |
| annotations | `/kaggle/input/datasets/freimutdiener/mc-phenobench-yolox/phenobench_multiclass_full_1024_raw/annotations` |
| └ provenance | `<annotations>/dataset_metadata.json` |
| COCO-pretrained checkpoint | `/kaggle/input/models/freimutdiener/yolox-nano-coco/pytorch/default/1/yolox_nano.pth` |

Both bundles carry the record of how they were built; section 5 audits the two
against each other rather than against constants written here.

The image bundle already carries the multiclass annotations it was exported alongside, so this notebook needs **one** Kaggle input.

Enable a **GPU accelerator** and **internet** (the setup cell clones YOLOX).

## Partial ("do-not-care") plants

The exporters follow the upstream PhenoBench protocol (`plant_visibility <=
0.5`): partials are **dropped from `train_annotations.json`** and **carried
flagged in the eval annotations**. Section 4 explains why that flag has to be
rewritten before `pycocotools` will honour it.

## Object scale at 320×320 — the dominant caveat

Box `sqrt(area)` after the resize to a 320 network input, measured on the
repository's `datasets/` mirror of this bundle. Treat it as indicative: the
mirror and the published Kaggle bundle have been observed to diverge (the
tiled multiclass export differs by ~1200 training annotations between the
two), so section 5 recomputes these figures from whatever is actually
attached, and those are the numbers to quote.

| p25 | p50 | p75 | < 8 px | < 16 px |
|---|---|---|---|---|
| 7.0 px | 16.1 px | 46.6 px | 29.6 % | 49.8 % |

YOLOX's finest FPN level is stride 8, so ground truth below 8 px occupies less
than one cell at the highest-resolution head and is effectively unassignable by
SimOTA. Section 5 recomputes this from the staged data rather than trusting
this table. Full frames are downscaled 1024 -> 320 (ratio 0.31), which is the harshest reduction of the four configurations; the tiled counterpart (notebook 17) exists precisely to quantify what that costs.

## Outputs

`YOLOX_outputs/yolox_nano_mc_phenobench_320/` (checkpoints, TensorBoard logs) and
`yolox_nano_mc_phenobench_320.onnx` (raw-output graph, decode excluded — see section 9).

## 1. Environment

YOLOX is installed from a **pinned commit**, not from `main`. Kaggle's
preinstalled PyTorch/CUDA build is left untouched: it is matched to the runtime
GPU image, and replacing it is the most common cause of a silently
CPU-only or ABI-broken run.

In [1]:
YOLOX_COMMIT = "6ddff4824372906469a7fae2dc3206c7aa4bbaee"   # pinned for reproducibility

import os
from pathlib import Path

WORK_ROOT = Path("/kaggle/working/yolox-phenobench")
YOLOX_ROOT = WORK_ROOT / "YOLOX"
WORK_ROOT.mkdir(parents=True, exist_ok=True)

if not (YOLOX_ROOT / "setup.py").exists():
    !git clone -q https://github.com/Megvii-BaseDetection/YOLOX.git "{YOLOX_ROOT}"
    !git -C "{YOLOX_ROOT}" checkout -q {YOLOX_COMMIT}

%cd "{YOLOX_ROOT}"
# requirements.txt already pins onnx>=1.13 and onnx-simplifier==0.4.10, which the
# export in section 9 needs. onnxruntime is extra: it is used to verify the
# exported graph, not to produce it.
!python -m pip install -q -r requirements.txt
!python -m pip install -q --no-build-isolation -e .
!python -m pip install -q onnxruntime

import torch

print("torch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("YOLOX:", YOLOX_ROOT, "@", YOLOX_COMMIT[:12])
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator before training."
print("GPU:", torch.cuda.get_device_name(0))

/kaggle/working/yolox-phenobench/YOLOX
torch: 2.0.0 | CUDA: 11.3
YOLOX: /kaggle/working/yolox-phenobench/YOLOX @ 6ddff4824372
GPU: Tesla P100-PCIE-16GB


## 2. Configuration

Every input path is written out in full. There is deliberately no fallback
search over candidate layouts: a missing input must fail loudly here, at second
0, rather than resolve to a plausible-but-wrong directory and surface as an
inexplicable metric three GPU-hours later.

In [2]:
# ---- Inputs (exact paths; see the header table) -------------------------
# The bundle root is named explicitly because the export writes its provenance
# record there (raw_images_metadata.json), and section 5 audits against it.
IMAGE_BUNDLE = Path("/kaggle/input/datasets/freimutdiener/mc-phenobench-yolox/phenobench_multiclass_full_1024_raw")
IMAGES_ROOT = IMAGE_BUNDLE / "images"
BUNDLE_METADATA = IMAGE_BUNDLE / "raw_images_metadata.json"

ANNOTATIONS_ROOT = Path("/kaggle/input/datasets/freimutdiener/mc-phenobench-yolox/phenobench_multiclass_full_1024_raw/annotations")
DATASET_METADATA = ANNOTATIONS_ROOT / "dataset_metadata.json"

PRETRAINED = Path("/kaggle/input/models/freimutdiener/yolox-nano-coco/pytorch/default/1/yolox_nano.pth")

# ---- Experiment identity ------------------------------------------------
EXP_NAME = "yolox_nano_mc_phenobench_320"
EXPECTED_CLASSES = ["crop", "weed"]
EXPECTED_TRAIN_IMAGES = 1407
EXPECTED_IMAGE_PX = 1024      # native size of a stored image

# ---- Training ------------------------------------------------------------
# 320 is the deployment size and the size the SSD arm uses; see the header.
IMAGE_SIZE = 320
BATCH_SIZE = 32
MAX_EPOCHS = 150
NO_AUG_EPOCHS = 15             # final epochs with augmentation off
NUM_WORKERS = 4
SEED = 42
# "ram" keeps the resized dataset in memory; "disk" memory-maps it under
# COCO_ROOT. Section 3 prints the actual footprint for this bundle.
CACHE_MODE = "ram"

# ---- Staged annotation file names ---------------------------------------
# The export's own names are kept; COCO's "instances_*2017.json" convention is
# not adopted, because nothing here is COCO-2017. Section 6 points the
# experiment at these explicitly, so no upstream default is relied on.
TRAIN_ANN = "train_annotations.json"
VAL_ANN = "val_annotations.json"                   # partials as do-not-care
VAL_ANN_STRICT = "val_annotations_strict.json"     # partials as ordinary GT
TEST_ANN = "test_annotations.json"
TEST_ANN_STRICT = "test_annotations_strict.json"

# ---- Derived working paths ----------------------------------------------
COCO_ROOT = WORK_ROOT / "phenobench_coco"
OUTPUT_ROOT = WORK_ROOT / "outputs"
EXP_FILE = YOLOX_ROOT / "exps" / "example" / "custom" / f"{EXP_NAME}.py"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Fail now, with the offending path, rather than later with a confusing error.
for role, path in [
    ("images", IMAGES_ROOT),
    ("image-bundle provenance", BUNDLE_METADATA),
    ("annotations", ANNOTATIONS_ROOT),
    ("annotation provenance", DATASET_METADATA),
    ("pretrained checkpoint", PRETRAINED),
]:
    assert path.exists(), (
        f"Missing {role}: {path}\n"
        "Attach the Kaggle input listed in the header table, or correct the "
        "constant above. Nothing is auto-discovered by design."
    )

print("images:     ", IMAGES_ROOT)
print("annotations:", ANNOTATIONS_ROOT)
print("pretrained: ", PRETRAINED)
print("experiment: ", EXP_NAME)

images:      /kaggle/input/datasets/freimutdiener/mc-phenobench-yolox/phenobench_multiclass_full_1024_raw/images
annotations: /kaggle/input/datasets/freimutdiener/mc-phenobench-yolox/phenobench_multiclass_full_1024_raw/annotations
pretrained:  /kaggle/input/models/freimutdiener/yolox-nano-coco/pytorch/default/1/yolox_nano.pth
experiment:  yolox_nano_mc_phenobench_320


## 3. Stage the COCO tree

`COCODataset` resolves images as `data_dir/<name>/<file_name>` and annotations
as `data_dir/annotations/<ann_file>`, and the export already uses bare
`file_name` entries. So the staged tree can mirror the bundle exactly:

```text
phenobench_coco/
├── images/        -> symlink to the bundle's images/
└── annotations/   copies of the bundle's *_annotations.json
```

**On the absent `2017`.** Upstream defaults the image directory to `train2017`
(`COCODataset(name="train2017")`) and hardcodes `val2017` in
`get_eval_dataset`, and its annotation files are conventionally
`instances_*2017.json`. None of that means anything here — it is COCO-2017
nomenclature, not a property of this dataset. The experiment in section 6
therefore overrides `name="images"` and points `train_ann` / `val_ann` at the
export's own file names, so no invented split directories or year suffixes
appear anywhere in the tree.

Images are **symlinked, never copied**: the bundle is read-only and several GB,
and Kaggle's working quota is finite. Annotations are copied because section 4
derives a second variant from them.

There is one image directory, not one per split: the bundle stores a single
flat `images/`, and the split is defined by the annotation files, not by
directory membership.

In [3]:
import json
import shutil

TRAIN_JSON = ANNOTATIONS_ROOT / "train_annotations.json"
VAL_JSON = ANNOTATIONS_ROOT / "val_annotations.json"
TEST_JSON = ANNOTATIONS_ROOT / "test_annotations.json"

for path in (TRAIN_JSON, VAL_JSON, TEST_JSON):
    assert path.is_file(), f"Missing annotation file: {path}"

shutil.rmtree(COCO_ROOT, ignore_errors=True)
(COCO_ROOT / "annotations").mkdir(parents=True)

# Mirror the export: one flat images/ beside annotations/. The experiment
# overrides COCODataset(name=...) to match, so no "train2017" fiction is needed.
os.symlink(IMAGES_ROOT.resolve(), COCO_ROOT / "images")

shutil.copy2(TRAIN_JSON, COCO_ROOT / "annotations" / TRAIN_ANN)

n_train = len(json.loads(TRAIN_JSON.read_text())["images"])
assert n_train == EXPECTED_TRAIN_IMAGES, (
    f"{TRAIN_JSON} has {n_train} images, expected {EXPECTED_TRAIN_IMAGES}. "
    "The annotation bundle does not match the image bundle geometry."
)

cache_gb = n_train * IMAGE_SIZE * IMAGE_SIZE * 3 / 1024**3
print(f"staged {n_train} training images")
print(f"--cache {CACHE_MODE} footprint at {IMAGE_SIZE}px: {cache_gb:.2f} GB")

staged 1407 training images
--cache ram footprint at 320px: 0.40 GB


## 4. Make the do-not-care flags effective

The eval annotations mark partially-visible plants with `ignore: 1` and
`partial: 1`, leaving `iscrowd: 0`. `agri_vision_edge`'s own evaluator reads
those flags (`evaluation.partials.is_partial_annotation`), but **`pycocotools`
does not**. Its `COCOeval._prepare` reads the field and then overwrites it:

```python
gt['ignore'] = gt['ignore'] if 'ignore' in gt else 0
gt['ignore'] = 'iscrowd' in gt and gt['iscrowd']   # <- discards the line above
```

So an unmodified eval file scores every partial as an ordinary positive: a
correct detection on a cut-off plant counts as a false positive, and a missed
one as a false negative. That is precisely the accounting the upstream
PhenoBench protocol excludes.

Two annotation sets are therefore staged, and both are reported in section 8:

- **strict** — as exported; partials are ordinary ground truth.
- **do-not-care** — `ignore: 1` promoted to `iscrowd: 1`, which is the channel
  `COCOeval` actually honours, making the number comparable to
  `ave evaluate --ignore-partials` and to the SSD arm.

Model selection uses the do-not-care set, since that is the reported metric.

In [4]:
def stage_eval_annotations(source: Path, strict_name: str, dnc_name: str) -> dict:
    """Write strict and do-not-care copies of one eval annotation file."""
    coco = json.loads(source.read_text())

    # Byte-faithful copy first: this is the ground truth `ave evaluate` and the
    # SSD arm see, and it must not be quietly improved here.
    (COCO_ROOT / "annotations" / strict_name).write_text(json.dumps(coco))

    n_partial = 0
    for ann in coco["annotations"]:
        # `partial` is this project's explicit marker; `ignore` is the COCO
        # standard one. Either implies do-not-care.
        if ann.get("partial") or ann.get("ignore"):
            ann["iscrowd"] = 1
            n_partial += 1

    (COCO_ROOT / "annotations" / dnc_name).write_text(json.dumps(coco))

    return {
        "source": source.name,
        "images": len(coco["images"]),
        "annotations": len(coco["annotations"]),
        "do_not_care": n_partial,
        "share": f"{n_partial / max(len(coco['annotations']), 1):.1%}",
    }


# The do-not-care copy takes the plain name because it is what the experiment
# validates against, so model selection optimises the reported metric.
for stats in (
    stage_eval_annotations(VAL_JSON, VAL_ANN_STRICT, VAL_ANN),
    stage_eval_annotations(TEST_JSON, TEST_ANN_STRICT, TEST_ANN),
):
    print(stats)

{'source': 'val_annotations.json', 'images': 386, 'annotations': 5170, 'do_not_care': 948, 'share': '18.3%'}
{'source': 'test_annotations.json', 'images': 386, 'annotations': 5238, 'do_not_care': 892, 'share': '17.0%'}


## 5. Integrity and scale audit

The export does not just ship images and annotations; it ships a record of how
they were made. `raw_images_metadata.json` sits at the image-bundle root and
`dataset_metadata.json` in the annotation root, and between them they pin the
tiling grid, the native image size, the class definition, the partial policy
and the split seed. Auditing against those records rather than against
constants hardcoded here is what makes this check meaningful: it compares the
two inputs *to each other*, not to my assumptions about them.

**Why this matters more than it looks.** Tiles are joined to their annotations
**by file name only**. A 2×2 grid and a 3×3 grid with `overlap=0.5` both
produce 512×512 tiles and both name them `<stem>_tile0..`, so pairing images
from one grid with annotations from another resolves every path, matches every
dimension, and scores every box against the wrong crop — silently. Every
weaker check (file existence, image dimensions, box validity) passes in that
scenario. Only comparing the two recorded grids catches it.

Here both records originate from the same export run, so the grids agree by construction and the assertion acts as a regression guard against a future bundle swap.

The scale table in the header was measured on the repository's dataset mirror
and is indicative only; the figures below are recomputed from the bundle that
is actually attached, and those are the ones to quote.

In [5]:
import math
from collections import Counter

# ---- 5a. Provenance: do the two bundles describe the same dataset? -------
bundle_meta = json.loads(BUNDLE_METADATA.read_text())
dataset_meta = json.loads(DATASET_METADATA.read_text())


def tiling_signature(record: dict):
    """Reduce either metadata dialect to a comparable (rows, cols, overlap)."""
    tiling = record.get("tiling")
    if tiling is None:
        return None
    return (int(tiling["rows"]), int(tiling["cols"]), float(tiling["overlap"]))


image_grid = tiling_signature(bundle_meta)
annotation_grid = tiling_signature(dataset_meta)
assert image_grid == annotation_grid, (
    f"Tiling grid mismatch.\n"
    f"  images were cut on      : {image_grid}\n"
    f"  annotations describe    : {annotation_grid}\n"
    "Tiles are joined to annotations by file name alone, and different grids "
    "can produce identical names and identical tile sizes. Left unchecked this "
    "trains against systematically wrong crops without raising anything."
)

declared = [
    c["name"]
    for c in sorted(dataset_meta["dataset_definition"]["categories"], key=lambda c: c["id"])
]
assert declared == EXPECTED_CLASSES, (
    f"{DATASET_METADATA} declares {declared}, this notebook expects {EXPECTED_CLASSES}."
)
assert int(dataset_meta["image_size"]) == EXPECTED_IMAGE_PX, (
    f"annotation bundle was built for {dataset_meta['image_size']}px images, "
    f"this notebook expects {EXPECTED_IMAGE_PX}px."
)
assert int(dataset_meta["train_samples"]) == EXPECTED_TRAIN_IMAGES
assert dataset_meta["partials_policy"] == "drop in train, do-not-care in eval", (
    f"unexpected partial policy: {dataset_meta['partials_policy']!r}"
)
assert bundle_meta["image_root"] == "images", bundle_meta["image_root"]

print("tiling grid          :", image_grid or "none (full frames)")
print("declared classes     :", declared)
print("native image size    :", dataset_meta["image_size"])
print("partial threshold    :", dataset_meta["partial_threshold"],
      "|", dataset_meta["partials_policy"])
print("split seed           :", dataset_meta["split_seed"])
print("images in bundle     :", bundle_meta["images_written_total"])
# For the single-class notebooks these two intentionally differ: the pixels come
# from the multiclass bundle, the labels from the single-class export. The grid
# assertion above is what makes that pairing safe.
print("images built against :", bundle_meta["source_annotation_artifacts"])
print("labels taken from    :", ANNOTATIONS_ROOT)


# ---- 5b. Structure of each staged split ---------------------------------
def audit(ann_file: str) -> dict:
    coco = json.loads((COCO_ROOT / "annotations" / ann_file).read_text())
    image_dir = COCO_ROOT / "images"

    sizes = {(i["width"], i["height"]) for i in coco["images"]}
    assert sizes == {(EXPECTED_IMAGE_PX, EXPECTED_IMAGE_PX)}, (
        f"{ann_file}: unexpected image geometry {sizes}; expected "
        f"{EXPECTED_IMAGE_PX}x{EXPECTED_IMAGE_PX}."
    )

    # Sample rather than stat() every file: the export notebooks already assert
    # full resolution of every reference, and these are symlinked.
    sample = coco["images"][:: max(len(coco["images"]) // 200, 1)]
    missing = [i["file_name"] for i in sample if not (image_dir / i["file_name"]).exists()]
    assert not missing, f"{ann_file}: unresolved image paths, e.g. {missing[:5]}"

    by_image = Counter()
    inverted, out_of_bounds, degenerate = [], [], []
    for ann in coco["annotations"]:
        x, y, w, h = ann["bbox"]
        by_image[ann["image_id"]] += 1
        if w < 0 or h < 0:
            # xywh cannot be negative. This is a format error, not clipping.
            inverted.append(ann["bbox"])
        elif x + w > EXPECTED_IMAGE_PX + 1 or y + h > EXPECTED_IMAGE_PX + 1:
            # If xyxy had been written into the xywh field, w would hold xmax
            # and x + w would land far outside the frame.
            out_of_bounds.append(ann["bbox"])
        elif w == 0 or h == 0:
            degenerate.append(ann["bbox"])

    assert not inverted, (
        f"{ann_file}: {len(inverted)} box(es) with negative width/height, "
        f"e.g. {inverted[:3]} — the bbox field is not valid COCO xywh."
    )
    assert not out_of_bounds, (
        f"{ann_file}: {len(out_of_bounds)} box(es) extend past the frame, "
        f"e.g. {out_of_bounds[:3]} — xyxy written into an xywh field?"
    )

    # Zero-area boxes are a border-clipping artefact of the exporter, not a
    # corrupt bundle: an instance whose mask survives filtering but collapses to
    # a single row or column at the frame/tile edge. They are reported, not
    # rejected, because YOLOX already discards them itself --
    # `if obj["area"] > 0 and x2 >= x1 and y2 >= y1` in COCODataset -- so they
    # never reach training. They are deliberately NOT stripped from the staged
    # annotations: that file is also the ground truth `ave evaluate` and the SSD
    # arm score against, and silently deleting ground truth here would flatter
    # this arm relative to that one. In evaluation they behave as ground truth
    # that no detection can ever match (IoU is identically 0), i.e. a small
    # fixed recall penalty applied equally to every detector scored on this file.
    max_labels = max(by_image.values()) if by_image else 0
    assert max_labels <= 50, (
        f"{ann_file}: an image carries {max_labels} annotations, but upstream "
        "TrainTransform(max_labels=50) truncates silently. Raise max_labels in "
        "the experiment's get_dataset override."
    )

    names = [c["name"] for c in sorted(coco["categories"], key=lambda c: c["id"])]
    return {
        "file": ann_file,
        "images": len(coco["images"]),
        "annotations": len(coco["annotations"]),
        "zero_area": len(degenerate),
        "max_labels_per_image": max_labels,
        "classes": names,
    }


reports = [audit(TRAIN_ANN), audit(VAL_ANN), audit(TEST_ANN)]
for report in reports:
    print(report)

total_degenerate = sum(r["zero_area"] for r in reports)
if total_degenerate:
    print(
        f"\nnote: {total_degenerate} zero-area box(es) across the three splits "
        "(border-clipped instances). Dropped by YOLOX before training; retained "
        "in the evaluation ground truth so this arm is scored on exactly the "
        "same annotations as the SSD arm."
    )

class_lists = {tuple(r["classes"]) for r in reports}
assert len(class_lists) == 1, f"Splits disagree on the class list: {class_lists}"

CLASS_NAMES = reports[0]["classes"]
NUM_CLASSES = len(CLASS_NAMES)
assert CLASS_NAMES == EXPECTED_CLASSES, (
    f"Class list {CLASS_NAMES} != expected {EXPECTED_CLASSES}. This notebook is "
    "wired to one bundle; check ANNOTATIONS_ROOT."
)

# --- scale at the network input -----------------------------------------
train = json.loads((COCO_ROOT / "annotations" / TRAIN_ANN).read_text())
ratio = IMAGE_SIZE / EXPECTED_IMAGE_PX
sides = sorted(math.sqrt(a["bbox"][2] * a["bbox"][3]) * ratio for a in train["annotations"])
n = len(sides)
pct = lambda p: sides[int(p * n)]
print(f"\nNUM_CLASSES = {NUM_CLASSES} {CLASS_NAMES}")
print(f"box sqrt(area) at {IMAGE_SIZE}px input (resize ratio {ratio:.3f}):")
print(f"  p25={pct(.25):.1f}  p50={pct(.50):.1f}  p75={pct(.75):.1f} px")
print(f"  below stride-8 cell: {sum(s < 8 for s in sides) / n:.1%}"
      f"   below 16 px: {sum(s < 16 for s in sides) / n:.1%}")
print("per-class boxes:", Counter(
    next(c["name"] for c in train["categories"] if c["id"] == a["category_id"])
    for a in train["annotations"]))

tiling grid          : none (full frames)
declared classes     : ['crop', 'weed']
native image size    : 1024
partial threshold    : 0.5 | drop in train, do-not-care in eval
split seed           : 42
images in bundle     : 2179
images built against : /kaggle/input/datasets/freimutdiener/mc-phenobench-no-partials
labels taken from    : /kaggle/input/datasets/freimutdiener/mc-phenobench-yolox/phenobench_multiclass_full_1024_raw/annotations
{'file': 'train_annotations.json', 'images': 1407, 'annotations': 16450, 'zero_area': 0, 'max_labels_per_image': 34, 'classes': ['crop', 'weed']}
{'file': 'val_annotations.json', 'images': 386, 'annotations': 5170, 'zero_area': 9, 'max_labels_per_image': 30, 'classes': ['crop', 'weed']}
{'file': 'test_annotations.json', 'images': 386, 'annotations': 5238, 'zero_area': 15, 'max_labels_per_image': 32, 'classes': ['crop', 'weed']}

note: 24 zero-area box(es) across the three splits (border-clipped instances). Dropped by YOLOX before training; retained in 

## 6. Experiment definition

The architecture is upstream `exps/default/yolox_nano.py` verbatim — depth
0.33, width 0.25, depthwise convolutions — so the only deviations from the
reference detector are those listed below. Each is a deliberate choice, not an
inherited default:

| setting | value | reason |
|---|---|---|
| `input_size` / `test_size` | 320 | deployment size; matches the SSD arm |
| `multiscale_range` | 2 | scale jitter 256–384, centred on the deployment size. Upstream's ±5 spans 160–480, which trains largely off-target and widens the activation range the INT8 calibration has to cover |
| `enable_mixup` | `False` | upstream nano default; MixUp composites whole scenes and is ill-suited to a two-class agricultural domain |
| `mosaic_prob` | 0.5 | upstream nano default. Mosaic is the main small-object augmentation, which matters here, but it also fabricates crop/weed adjacencies that do not occur in the field — hence 0.5 rather than 1.0 |
| `seed` | 42 | `train.py` reads `exp.seed`; there is **no `--seed` CLI flag** |
| `eval_interval` | 5 | model selection resolution |

`self.seed` is set here rather than passed on the command line because
`tools/train.py` only consults `exp.seed`. Setting it also enables cuDNN
deterministic mode upstream, which costs throughput but makes the run
repeatable.

In [6]:
# Tokens are substituted textually rather than with an f-string so the Python
# below stays valid, readable, and copy-pasteable into the YOLOX tree.
EXP_TEMPLATE = """
# PhenoBench YOLOX-Nano experiment (generated by
# scripts/gen_yolox_finetune_notebooks.py; do not edit in place).

import os

import torch.nn as nn

from yolox.exp import Exp as MyExp


class Exp(MyExp):
    def __init__(self):
        super(Exp, self).__init__()

        # --- architecture: upstream yolox_nano.py ---------------------------
        self.depth = 0.33
        self.width = 0.25

        # --- dataset --------------------------------------------------------
        self.num_classes = __NUM_CLASSES__
        self.data_dir = r"__COCO_ROOT__"
        self.train_ann = "__TRAIN_ANN__"
        self.val_ann = "__VAL_ANN__"

        # --- resolution -----------------------------------------------------
        self.input_size = (__IMAGE_SIZE__, __IMAGE_SIZE__)
        self.test_size = (__IMAGE_SIZE__, __IMAGE_SIZE__)
        # random_size is derived as input_size/32 +/- multiscale_range.
        self.multiscale_range = 2

        # --- schedule -------------------------------------------------------
        self.max_epoch = __MAX_EPOCHS__
        self.no_aug_epochs = __NO_AUG_EPOCHS__
        self.warmup_epochs = 3
        self.eval_interval = 5
        self.data_num_workers = __NUM_WORKERS__
        self.basic_lr_per_img = 0.01 / 64.0
        self.min_lr_ratio = 0.05
        self.ema = True

        # --- augmentation ---------------------------------------------------
        self.mosaic_prob = 0.5
        self.mosaic_scale = (0.5, 1.5)
        self.enable_mixup = False
        self.mixup_prob = 0.0
        self.hsv_prob = 1.0
        self.flip_prob = 0.5
        self.degrees = 10.0
        self.translate = 0.1
        self.shear = 2.0

        # --- reproducibility: tools/train.py reads exp.seed -----------------
        self.seed = __SEED__

        self.exp_name = os.path.split(os.path.realpath(__file__))[1].split(".")[0]

    # ------------------------------------------------------------------
    # Dataset directory name.
    #
    # Upstream defaults COCODataset(name="train2017") and hardcodes
    # name="val2017" in get_eval_dataset, i.e. it assumes COCO-2017's
    # one-directory-per-split layout. This bundle stores a single flat
    # images/ directory, so both overrides pass name="images" and the
    # staged tree needs no invented split directories. Everything else is
    # upstream's implementation unchanged.
    # ------------------------------------------------------------------

    def get_dataset(self, cache: bool = False, cache_type: str = "ram"):
        from yolox.data import COCODataset, TrainTransform

        return COCODataset(
            data_dir=self.data_dir,
            json_file=self.train_ann,
            name="images",
            img_size=self.input_size,
            preproc=TrainTransform(
                max_labels=50,
                flip_prob=self.flip_prob,
                hsv_prob=self.hsv_prob,
            ),
            cache=cache,
            cache_type=cache_type,
        )

    def get_eval_dataset(self, **kwargs):
        from yolox.data import COCODataset, ValTransform

        # The upstream testdev branch is dropped: there is no test-dev split
        # here, and the held-out test half is selected by overriding val_ann
        # on the command line instead.
        return COCODataset(
            data_dir=self.data_dir,
            json_file=self.val_ann,
            name="images",
            img_size=self.test_size,
            preproc=ValTransform(legacy=kwargs.get("legacy", False)),
        )

    def get_model(self, sublinear=False):
        def init_yolo(M):
            for m in M.modules():
                if isinstance(m, nn.BatchNorm2d):
                    m.eps = 1e-3
                    m.momentum = 0.03

        if "model" not in self.__dict__:
            from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead
            in_channels = [256, 512, 1024]
            # depthwise=True is the defining difference of the Nano variant.
            backbone = YOLOPAFPN(
                self.depth, self.width, in_channels=in_channels,
                act=self.act, depthwise=True,
            )
            head = YOLOXHead(
                self.num_classes, self.width, in_channels=in_channels,
                act=self.act, depthwise=True,
            )
            self.model = YOLOX(backbone, head)

        self.model.apply(init_yolo)
        self.model.head.initialize_biases(1e-2)
        return self.model
"""

EXP_FILE.parent.mkdir(parents=True, exist_ok=True)
EXP_FILE.write_text(
    EXP_TEMPLATE
    .replace("__NUM_CLASSES__", str(NUM_CLASSES))
    .replace("__COCO_ROOT__", str(COCO_ROOT))
    .replace("__TRAIN_ANN__", TRAIN_ANN)
    .replace("__VAL_ANN__", VAL_ANN)
    .replace("__IMAGE_SIZE__", str(IMAGE_SIZE))
    .replace("__MAX_EPOCHS__", str(MAX_EPOCHS))
    .replace("__NO_AUG_EPOCHS__", str(NO_AUG_EPOCHS))
    .replace("__NUM_WORKERS__", str(NUM_WORKERS))
    .replace("__SEED__", str(SEED))
)
print(EXP_FILE)
print(EXP_FILE.read_text())

/kaggle/working/yolox-phenobench/YOLOX/exps/example/custom/yolox_nano_mc_phenobench_320.py

# PhenoBench YOLOX-Nano experiment (generated by
# scripts/gen_yolox_finetune_notebooks.py; do not edit in place).

import os

import torch.nn as nn

from yolox.exp import Exp as MyExp


class Exp(MyExp):
    def __init__(self):
        super(Exp, self).__init__()

        # --- architecture: upstream yolox_nano.py ---------------------------
        self.depth = 0.33
        self.width = 0.25

        # --- dataset --------------------------------------------------------
        self.num_classes = 2
        self.data_dir = r"/kaggle/working/yolox-phenobench/phenobench_coco"
        self.train_ann = "train_annotations.json"
        self.val_ann = "val_annotations.json"

        # --- resolution -----------------------------------------------------
        self.input_size = (320, 320)
        self.test_size = (320, 320)
        # random_size is derived as input_size/32 +/- multiscale_range.
     

## 7. Train

`-b` is the **total** batch size and `-d 1` states the single Kaggle GPU
explicitly. `-c` supplies COCO-pretrained weights: YOLOX loads the backbone and
neck and re-initialises the head for this class count, which matters on a
dataset this small.

`--cache` resizes the dataset once and reuses it. Without it the run is
dataloader-bound: mosaic reads four images per sample, and these are PNGs at
1024 px against a small Kaggle vCPU allocation.

Note there is **no `--seed`**; reproducibility comes from `exp.seed`, set in
section 6.

In [7]:
%cd "{YOLOX_ROOT}"
!python tools/train.py \
  -f "{EXP_FILE}" \
  -d 1 \
  -b {BATCH_SIZE} \
  --fp16 \
  -o \
  --cache {CACHE_MODE} \
  -c "{PRETRAINED}"

/kaggle/working/yolox-phenobench/YOLOX
/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
loading annotations into memory...
Done (t=0.05s)
creating index...
index created!
2026-08-06 10:49:58.118 | INFO     | yolox.data.datasets.datasets_wrapper:cache_images:211 - 0.4GB RAM required, 29.5/31.3GB RAM available, Since the first thing we do is cache, there is no guarantee that the remaining memory space is sufficient
2026-08-06 10:49:58.118 | INFO     | yolox.data.datasets.datasets_wrapper:cache_images:221 - You are using cached images in RAM to accelerate training!
2026-08-06 10:49:58.119 | INFO     | yolox.data.datasets.datasets_wrapper:cache_images:237 - Caching images...
This might take some time for your dataset
Caching images (0.4/0.4GB ram): 100%|███████| 1407/1407 [00:24<00:00, 57.87i

## 8. Evaluate

Three numbers are produced:

1. **val, do-not-care** — the model-selection metric, comparable to
   `ave evaluate --ignore-partials`.
2. **val, strict** — the same checkpoint scored with partials as ordinary
   ground truth. The gap between 1 and 2 quantifies how much of the score is
   decided by border plants, which is worth stating explicitly rather than
   leaving as an unexamined protocol difference.
3. **test, do-not-care** — the held-out half of the official validation split,
   touched once.

`tools/eval.py` accepts trailing `key value` pairs that `Exp.merge` applies, so
the split is switched with `val_ann <file>` and no second experiment file is
needed. These numbers are still YOLOX-internal: the figure that enters the
thesis comparison comes from `ave evaluate` on exported predictions, under the
same NMS and score threshold as the SSD arm.

In [8]:
RUN_DIR = YOLOX_ROOT / "YOLOX_outputs" / EXP_NAME
BEST_CKPT = RUN_DIR / "best_ckpt.pth"

if not BEST_CKPT.exists():
    # An interrupted run still leaves latest_ckpt.pth; fall back so the export
    # path can be exercised, but say so loudly.
    written = sorted(RUN_DIR.glob("*.pth"), key=lambda p: p.stat().st_mtime)
    assert written, f"No checkpoints in {RUN_DIR}"
    BEST_CKPT = written[-1]
    print(f"WARNING: best_ckpt.pth absent, falling back to {BEST_CKPT.name}")

print("checkpoints:", [p.name for p in sorted(RUN_DIR.glob("*.pth"))])
print("evaluating: ", BEST_CKPT)

checkpoints: ['best_ckpt.pth', 'epoch_100_ckpt.pth', 'epoch_105_ckpt.pth', 'epoch_10_ckpt.pth', 'epoch_110_ckpt.pth', 'epoch_115_ckpt.pth', 'epoch_120_ckpt.pth', 'epoch_125_ckpt.pth', 'epoch_130_ckpt.pth', 'epoch_135_ckpt.pth', 'epoch_136_ckpt.pth', 'epoch_137_ckpt.pth', 'epoch_138_ckpt.pth', 'epoch_139_ckpt.pth', 'epoch_140_ckpt.pth', 'epoch_141_ckpt.pth', 'epoch_142_ckpt.pth', 'epoch_143_ckpt.pth', 'epoch_144_ckpt.pth', 'epoch_145_ckpt.pth', 'epoch_146_ckpt.pth', 'epoch_147_ckpt.pth', 'epoch_148_ckpt.pth', 'epoch_149_ckpt.pth', 'epoch_150_ckpt.pth', 'epoch_15_ckpt.pth', 'epoch_20_ckpt.pth', 'epoch_25_ckpt.pth', 'epoch_30_ckpt.pth', 'epoch_35_ckpt.pth', 'epoch_40_ckpt.pth', 'epoch_45_ckpt.pth', 'epoch_50_ckpt.pth', 'epoch_55_ckpt.pth', 'epoch_5_ckpt.pth', 'epoch_60_ckpt.pth', 'epoch_65_ckpt.pth', 'epoch_70_ckpt.pth', 'epoch_75_ckpt.pth', 'epoch_80_ckpt.pth', 'epoch_85_ckpt.pth', 'epoch_90_ckpt.pth', 'epoch_95_ckpt.pth', 'last_epoch_ckpt.pth', 'last_mosaic_epoch_ckpt.pth', 'latest_ckpt

In [9]:
%cd "{YOLOX_ROOT}"
print("=" * 70, "\nval / do-not-care (model-selection metric)\n", "=" * 70)
!python tools/eval.py -f "{EXP_FILE}" -c "{BEST_CKPT}" -d 1 -b {BATCH_SIZE} \
  --fp16 --conf 0.001 --nms 0.65

/kaggle/working/yolox-phenobench/YOLOX
val / do-not-care (model-selection metric)
/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
2026-08-06 11:22:46 | INFO     | __main__:139 - Args: Namespace(experiment_name='yolox_nano_mc_phenobench_320', name=None, dist_backend='nccl', dist_url=None, batch_size=32, devices=1, num_machines=1, machine_rank=0, exp_file='/kaggle/working/yolox-phenobench/YOLOX/exps/example/custom/yolox_nano_mc_phenobench_320.py', ckpt='/kaggle/working/yolox-phenobench/YOLOX/YOLOX_outputs/yolox_nano_mc_phenobench_320/best_ckpt.pth', conf=0.001, nms=0.65, tsize=None, seed=None, fp16=True, fuse=False, trt=False, legacy=False, test=False, speed=False, opts=[])
2026-08-06 11:22:46 | INFO     | __main__:149 - Model Summary: Params: 0.90M, Gflops: 0.64
2026-08-06 11:22:46 | INFO

In [10]:
%cd "{YOLOX_ROOT}"
print("=" * 70, "\nval / strict (partials scored as ordinary ground truth)\n", "=" * 70)
!python tools/eval.py -f "{EXP_FILE}" -c "{BEST_CKPT}" -d 1 -b {BATCH_SIZE} \
  --fp16 --conf 0.001 --nms 0.65 \
  val_ann {VAL_ANN_STRICT}

print("=" * 70, "\ntest / do-not-care (held out)\n", "=" * 70)
!python tools/eval.py -f "{EXP_FILE}" -c "{BEST_CKPT}" -d 1 -b {BATCH_SIZE} \
  --fp16 --conf 0.001 --nms 0.65 \
  val_ann {TEST_ANN}

/kaggle/working/yolox-phenobench/YOLOX
val / strict (partials scored as ordinary ground truth)
/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
2026-08-06 11:23:02 | INFO     | __main__:139 - Args: Namespace(experiment_name='yolox_nano_mc_phenobench_320', name=None, dist_backend='nccl', dist_url=None, batch_size=32, devices=1, num_machines=1, machine_rank=0, exp_file='/kaggle/working/yolox-phenobench/YOLOX/exps/example/custom/yolox_nano_mc_phenobench_320.py', ckpt='/kaggle/working/yolox-phenobench/YOLOX/YOLOX_outputs/yolox_nano_mc_phenobench_320/best_ckpt.pth', conf=0.001, nms=0.65, tsize=None, seed=None, fp16=True, fuse=False, trt=False, legacy=False, test=False, speed=False, opts=['val_ann', 'val_annotations_strict.json'])
2026-08-06 11:23:03 | INFO     | __main__:149 - Model Summary: P

## 9. Export ONNX

Two corrections against the obvious invocation, both verified against
`tools/export_onnx.py` at the pinned commit:

**There is no `-s` flag.** The exporter takes its input shape from the
experiment:

```python
dummy_input = torch.randn(args.batch_size, 3, exp.test_size[0], exp.test_size[1])
```

Passing `-s 320` fails with `error: unrecognized arguments: -s`. Worse, `opts`
is declared `nargs=argparse.REMAINDER`, so the first unrecognised token
swallows everything after it — any later flag is silently discarded rather than
rejected. `test_size` is set in section 6, so the shape is already correct.

**`--decode_in_inference` is deliberately omitted.** The flag sets
`model.head.decode_in_inference`, which folds the grid/stride decode into the
graph. That decode contains `exp()` on the width/height branch: an unbounded
activation that INT8 calibration cannot represent without destroying the range
of everything sharing its scale. Leaving it out yields raw head outputs of
shape `[1, n_anchors, 5 + num_classes]` and moves the decode to the host —
which is also what this project already does for SSD (`saved_model_nms0`), so
both arms are exported on the same footing.

`replace_module(model, nn.SiLU, SiLU)` runs unconditionally upstream, rewriting
SiLU as `x * sigmoid(x)`; there is no native TFLite SiLU op, so this is
required for the downstream conversion regardless.

In [11]:
ONNX_FILE = OUTPUT_ROOT / f"{EXP_NAME}.onnx"

%cd "{YOLOX_ROOT}"
!python tools/export_onnx.py \
  -f "{EXP_FILE}" \
  -c "{BEST_CKPT}" \
  --output-name "{ONNX_FILE}" \
  --input images \
  --output output \
  --batch-size 1 \
  --opset 13

assert ONNX_FILE.is_file(), "ONNX export did not produce a file."
print("wrote", ONNX_FILE, f"({ONNX_FILE.stat().st_size / 1e6:.1f} MB)")

/kaggle/working/yolox-phenobench/YOLOX
2026-08-06 11:23:34.965 | INFO     | __main__:main:64 - args value: Namespace(output_name='/kaggle/working/yolox-phenobench/outputs/yolox_nano_mc_phenobench_320.onnx', input='images', output='output', opset=13, batch_size=1, dynamic=False, no_onnxsim=False, exp_file='/kaggle/working/yolox-phenobench/YOLOX/exps/example/custom/yolox_nano_mc_phenobench_320.py', experiment_name=None, name=None, ckpt='/kaggle/working/yolox-phenobench/YOLOX/YOLOX_outputs/yolox_nano_mc_phenobench_320/best_ckpt.pth', opts=[], decode_in_inference=False)
2026-08-06 11:23:35.057 | INFO     | __main__:main:88 - loading checkpoint done.
/kaggle/working/yolox-phenobench/YOLOX/tools/export_onnx.py:91: FutureWarning: 'torch.onnx._export' is deprecated in version 1.12.0 and will be removed in 2.0. Please use `torch.onnx.export` instead.
  torch.onnx._export(
================ Diagnostic Run torch.onnx.export version 2.0.0 ================
verbose: False, log level: Level.ERROR
====

### Verify the exported graph

An ONNX file that loads is not evidence of a correct export. This runs the
graph under `onnxruntime` on a real validation image and decodes it on the host
exactly as the deployment will, which is the only check that distinguishes a
correct raw-output export from a decoded one — the two differ in the *values*
of the box channels, not in the shape.

The operator histogram is also the input to the TFLite/NPU triage: `Resize`,
`Slice`/`Concat` (the `Focus` stem) and `Sigmoid`+`Mul` (SiLU) are the usual
delegation boundaries on the i.MX targets.

In [12]:
from collections import Counter

import cv2
import numpy as np
import onnx
import onnxruntime

from yolox.data.data_augment import preproc as preprocess
from yolox.utils import demo_postprocess, multiclass_nms

model = onnx.load(str(ONNX_FILE))
onnx.checker.check_model(model)

graph_input = model.graph.input[0]
shape = [d.dim_value or d.dim_param for d in graph_input.type.tensor_type.shape.dim]
print("opset:", model.opset_import[0].version, "| input:", graph_input.name, shape)

ops = Counter(node.op_type for node in model.graph.node)
print("nodes:", sum(ops.values()))
print("op histogram:", dict(ops.most_common()))
for op in ("Resize", "Slice", "Concat", "Sigmoid", "Mul", "Exp"):
    if ops.get(op):
        print(f"  {op:8s} x{ops[op]:4d}   <- review for INT8 / NPU delegation")
assert not ops.get("Exp"), (
    "Exp present: the decode was folded into the graph. Re-export without "
    "--decode_in_inference."
)

# --- run the graph on a real image, decode on the host -------------------
val = json.loads((COCO_ROOT / "annotations" / VAL_ANN).read_text())
sample = val["images"][0]
bgr = cv2.imread(str(COCO_ROOT / "images" / sample["file_name"]))
tensor, ratio = preprocess(bgr, (IMAGE_SIZE, IMAGE_SIZE))

session = onnxruntime.InferenceSession(str(ONNX_FILE), providers=["CPUExecutionProvider"])
raw = session.run(None, {session.get_inputs()[0].name: tensor[None, :, :, :]})[0]
print("\nraw output:", raw.shape, "expected (1, n_anchors, %d)" % (5 + NUM_CLASSES))
assert raw.shape[2] == 5 + NUM_CLASSES

# Host-side decode: grid + stride reconstruction, then score filter and NMS.
predictions = demo_postprocess(raw, (IMAGE_SIZE, IMAGE_SIZE))[0]
boxes, scores = predictions[:, :4], predictions[:, 4:5] * predictions[:, 5:]
xyxy = np.empty_like(boxes)
xyxy[:, 0] = boxes[:, 0] - boxes[:, 2] / 2
xyxy[:, 1] = boxes[:, 1] - boxes[:, 3] / 2
xyxy[:, 2] = boxes[:, 0] + boxes[:, 2] / 2
xyxy[:, 3] = boxes[:, 1] + boxes[:, 3] / 2
detections = multiclass_nms(xyxy / ratio, scores, nms_thr=0.65, score_thr=0.30)

n_gt = sum(a["image_id"] == sample["id"] for a in val["annotations"])
print(f"{sample['file_name']}: {0 if detections is None else len(detections)} "
      f"detections @0.30 vs {n_gt} ground-truth boxes")
if detections is not None and len(detections):
    for *box, score, cls in detections[:5]:
        print(f"  {CLASS_NAMES[int(cls)]:6s} {score:.3f} "
              f"[{box[0]:7.1f} {box[1]:7.1f} {box[2]:7.1f} {box[3]:7.1f}]")
    assert detections[:, 4].max() <= 1.0, "Scores out of range; decode is wrong."
    print("\nGraph verified: raw outputs decode to plausible boxes.")

opset: 13 | input: images [1, 3, 320, 320]
nodes: 367
op histogram: {'Conv': 113, 'Sigmoid': 110, 'Mul': 104, 'Concat': 18, 'Add': 7, 'Slice': 6, 'MaxPool': 3, 'Reshape': 3, 'Resize': 2, 'Transpose': 1}
  Resize   x   2   <- review for INT8 / NPU delegation
  Slice    x   6   <- review for INT8 / NPU delegation
  Concat   x  18   <- review for INT8 / NPU delegation
  Sigmoid  x 110   <- review for INT8 / NPU delegation
  Mul      x 104   <- review for INT8 / NPU delegation

raw output: (1, 2100, 7) expected (1, n_anchors, 7)
05-15_00259_P0030792.png: 7 detections @0.30 vs 9 ground-truth boxes
  crop   0.971 [   85.3   569.3   251.3   764.9]
  crop   0.963 [   81.3   803.3   249.1   962.1]
  crop   0.774 [  135.6   438.0   168.7   454.5]
  crop   0.741 [  130.1   209.5   143.7   232.7]
  weed   0.632 [  176.9   746.7   204.6   780.3]

Graph verified: raw outputs decode to plausible boxes.


## 10. Artifacts

| artifact | path |
|---|---|
| checkpoints + TensorBoard | `YOLOX_outputs/yolox_nano_mc_phenobench_320/` |
| best checkpoint | `YOLOX_outputs/yolox_nano_mc_phenobench_320/best_ckpt.pth` |
| ONNX (raw outputs, 1×3×320×320) | `outputs/yolox_nano_mc_phenobench_320.onnx` |

Save the run as a Kaggle notebook output so `scripts/sync_kaggle_runs.py` can
pick it up.

### Carrying this to INT8 TFLite

The ONNX graph is deliberately decode-free, so the conversion target is the
backbone/neck/head only, and the grid decode plus NMS stay on the host — the
same split as the SSD arm's `saved_model_nms0`. Calibration must use images
drawn from **this** bundle at 320×320; `rep_dataset.json` in the annotation
bundle lists 200 training indices, which map to `train_annotations.json`'s
`images[i]` because the exporter enumerates `range(len(dataset))` with
`image_id = i + 1`.

Two architecture-level risks are visible in the operator histogram above and
are worth settling before investing in the conversion: the `Focus` stem
(`Slice`/`Concat`) and SiLU (`Sigmoid`+`Mul`). Both are common delegation
boundaries on the i.MX8MP VX delegate and the i.MX93 Ethos-U55; `act="relu"`
and a plain convolutional stem are the usual remedies, at a measurable accuracy
cost that belongs in the ablation table rather than in an unexamined default.